# Bronze Layer — Raw Ingestion

Ingests the two transactional source files as-is into Delta tables with no transformation.
Schema enforcement is intentionally minimal — the goal is to preserve raw data exactly
as received, so Silver always has a full audit trail to recover from if cleaning logic changes.

Geographic reference files (`post_codes.geojson`, `amsterdam_areas.geojson`) are intentionally
excluded from Bronze. They are already clean, require no transformation, and are consumed
directly as raw assets by the Silver layer's zipcode backfill. Routing them through Delta
would add complexity without adding value.

In [0]:
# Databricks: cluster must have Delta Lake runtime (included by default)
# Local: pip install pyspark delta-spark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime, timezone

spark = SparkSession.builder.appName("amsterdam-rental-bronze").getOrCreate()

try:
    dbutils.fs.ls("/")  # only exists on Databricks
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    nb_path = ctx.notebookPath().get()  # e.g. /Repos/email/repo/notebooks/01_bronze
    repo_ws_root = "/Workspace" + "/".join(nb_path.split("/")[:-2])
    DATA_ROOT   = f"{repo_ws_root}/data"
    OUTPUT_ROOT = "/Volumes/workspace/default/amsterdam_rental/output"
except NameError:
    import os
    _repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
    DATA_ROOT   = f"{_repo}/data"
    OUTPUT_ROOT = f"{_repo}/data/output"

AIRBNB_SRC  = f"{DATA_ROOT}/airbnb.csv"
RENTALS_SRC = f"{DATA_ROOT}/rentals.json"
BRONZE_PATH = f"{OUTPUT_ROOT}/bronze"

INGESTED_AT = datetime.now(timezone.utc).isoformat()

## Airbnb — bronze

In [0]:
import os, shutil, zipfile

# Spark on serverless cannot read /Workspace paths, and the repo ships the
# source as airbnb.zip — stage the CSV into the UC volume before reading.
AIRBNB_READ_PATH = f"{OUTPUT_ROOT}/source/airbnb.csv"
os.makedirs(os.path.dirname(AIRBNB_READ_PATH), exist_ok=True)
if not os.path.exists(AIRBNB_READ_PATH):
    if os.path.exists(AIRBNB_SRC):
        shutil.copy(AIRBNB_SRC, AIRBNB_READ_PATH)
    else:
        with zipfile.ZipFile(AIRBNB_SRC.replace(".csv", ".zip")) as zf:
            with zf.open("airbnb.csv") as src, open(AIRBNB_READ_PATH, "wb") as dst:
                shutil.copyfileobj(src, dst)

airbnb_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")   # keep everything as strings at bronze
    .csv(AIRBNB_READ_PATH)
    .withColumn("_ingested_at", F.lit(INGESTED_AT))
    .withColumn("_source_file", F.lit("airbnb.csv"))
)

print(f"Airbnb rows: {airbnb_raw.count()}")
airbnb_raw.printSchema()
airbnb_raw.show(3, truncate=False)

In [0]:
(
    airbnb_raw
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{BRONZE_PATH}/airbnb")
)
print("Airbnb bronze written.")

## Rentals — bronze

In [0]:
import os, shutil, zipfile

# Spark on serverless cannot read /Workspace paths, and the repo ships the
# source as rentals.zip — stage the JSON into the UC volume before reading.
RENTALS_READ_PATH = f"{OUTPUT_ROOT}/source/rentals.json"
os.makedirs(os.path.dirname(RENTALS_READ_PATH), exist_ok=True)
if not os.path.exists(RENTALS_READ_PATH):
    if os.path.exists(RENTALS_SRC):
        shutil.copy(RENTALS_SRC, RENTALS_READ_PATH)
    else:
        with zipfile.ZipFile(RENTALS_SRC.replace(".json", ".zip")) as zf:
            with zf.open("rentals.json") as src, open(RENTALS_READ_PATH, "wb") as dst:
                shutil.copyfileobj(src, dst)

rentals_raw = (
    spark.read
    .option("multiline", "true")
    .json(RENTALS_READ_PATH)
    .withColumn("_ingested_at", F.lit(INGESTED_AT))
    .withColumn("_source_file", F.lit("rentals.json"))
)

print(f"Rentals rows: {rentals_raw.count()}")
rentals_raw.printSchema()
rentals_raw.show(3, truncate=True)

In [0]:
(
    rentals_raw
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{BRONZE_PATH}/rentals")
)
print("Rentals bronze written.")

## Validation checks

In [0]:
airbnb_bronze  = spark.read.format("delta").load(f"{BRONZE_PATH}/airbnb")
rentals_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/rentals")

assert airbnb_bronze.count()  == 9913,  "Unexpected Airbnb row count"
assert rentals_bronze.count() == 46722, "Unexpected Rentals row count"

print("Bronze validation passed.")